In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Importing Dependencies

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

In [3]:
# Config
DATA = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ["A", "B", "C", "D", "E"]
VEC_PARAMS = dict(max_features=50000, ngram_range=(1, 2), stop_words="english")

# Load
train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
print("train:", train.shape, "| test:", test.shape)

# Helpers
def make_text(df):
    """Combine the prompt with all 5 options into a single string per row."""
    return (df["prompt"]
            + " A: " + df["A"] + " B: " + df["B"] + " C: " + df["C"]
            + " D: " + df["D"] + " E: " + df["E"])

def map3_from_proba(probs, classes, truth):
    """Mean Average Precision @ 3 from predicted class probabilities."""
    total = 0.0
    for p, t in zip(probs, truth):
        top3 = classes[np.argsort(p)[::-1][:3]]
        for rank, lab in enumerate(top3):
            if lab == t:
                total += 1.0 / (rank + 1)
                break
    return total / len(truth)

X = make_text(train)
y = train["answer"].values

train: (2000, 8) | test: (500, 7)


# 1. Holdout validation

In [4]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([("tfidf", TfidfVectorizer(**VEC_PARAMS)),
                 ("clf",   LogisticRegression(max_iter=1000))])
pipe.fit(X_tr, y_tr)
holdout_map3 = map3_from_proba(pipe.predict_proba(X_va), pipe.classes_, y_va)
print(f"Holdout MAP@3      : {holdout_map3:.4f}")

Holdout MAP@3      : 1.0000


# 2. 5-fold cross-validation

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for tri, vai in skf.split(X, y):
    m = Pipeline([("tfidf", TfidfVectorizer(**VEC_PARAMS)),
                  ("clf",   LogisticRegression(max_iter=1000))]).fit(X.iloc[tri], y[tri])
    cv_scores.append(map3_from_proba(m.predict_proba(X.iloc[vai]), m.classes_, y[vai]))
cv_mean, cv_std = float(np.mean(cv_scores)), float(np.std(cv_scores))
print(f"5-fold CV MAP@3    : {cv_mean:.4f} +/- {cv_std:.4f}")

5-fold CV MAP@3    : 1.0000 +/- 0.0000


# 3. Artifact analysis

In [6]:
opt_text, opt_lab = [], []
for _, r in train.iterrows():
    for o in OPTIONS:
        opt_text.append(str(r[o]))
        opt_lab.append(1 if r["answer"] == o else 0)
av = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
Xo = av.fit_transform(opt_text)
Xo_tr, Xo_va, yo_tr, yo_va = train_test_split(
    Xo, np.array(opt_lab), test_size=0.2, random_state=42, stratify=opt_lab)
ac = LogisticRegression(max_iter=1000).fit(Xo_tr, yo_tr)
distractor_auc = roc_auc_score(yo_va, ac.predict_proba(Xo_va)[:, 1])
print(f"Distractor AUC     : {distractor_auc:.4f}  (artifact: distractors are detectable)")

Distractor AUC     : 0.9789  (artifact: distractors are detectable)


# 4. Final fit + submission

In [7]:
final = Pipeline([("tfidf", TfidfVectorizer(**VEC_PARAMS)),
                  ("clf",   LogisticRegression(max_iter=1000))]).fit(X, y)
probs = final.predict_proba(make_text(test))
preds = [" ".join(final.classes_[np.argsort(p)[::-1][:3]]) for p in probs]
submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")

submission.csv saved


# 5. Weights & Biases

In [8]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

run = wandb.init(
    entity="ishankgpt02-na",
    project="23f1002033-t22026",
    name="tfidf-lr-supervised",
    config={"model": "TF-IDF + LogisticRegression"},
)

wandb.log({
    "holdout_map3": holdout_map3,
    "cv_map3": cv_mean,
    "distractor_auc": distractor_auc,
})

run.finish()
print("Logged to W&B")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260629_185947-elybnp8w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-lr-supervised
wandb: ⭐️ View project at https://wandb.ai/ishankgpt02-na/23f1002033-t22026
wandb: 🚀 View run at https://wandb.ai/ishankgpt02-na/23f1002033-t22026/runs/elybnp8w
wandb: updating run metadata; uploading summary
wandb: uploading history 

Logged to W&B
